# Coffee Prices and Weather Visualizations

This notebook explores the relationship between weather patterns and coffee commodity prices using `Coffee_Data_Set.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

ROOT = Path.cwd()
if not (ROOT / 'Coffee_Data_Set.csv').exists():
    ROOT = ROOT.parent

DATASET_PATH = ROOT / 'Coffee_Data_Set.csv'
df = pd.read_csv(DATASET_PATH, parse_dates=['Date'])

weather_columns = [
    'Temp_Max',
    'Temp_Min',
    'Humidity',
    'Solar_Radiation',
    'Precipitation_mm',
]

df[weather_columns + ['Close_USD_60kg', 'brl', 'cny', 'mxn']] = df[
    weather_columns + ['Close_USD_60kg', 'brl', 'cny', 'mxn']
].apply(pd.to_numeric, errors='coerce')

df.head()

In [ ]:
print(f'Rows: {len(df):,}')
print(f'Locations: {df["Location"].nunique():,}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')

df[weather_columns + ['Close_USD_60kg']].describe().T

## Prepare Daily Aggregates

In [ ]:
daily = (
    df.groupby('Date', as_index=False)
    .agg({
        'Close_USD_60kg': 'first',
        'Temp_Max': 'mean',
        'Temp_Min': 'mean',
        'Humidity': 'mean',
        'Solar_Radiation': 'mean',
        'Precipitation_mm': 'mean',
        'brl': 'first',
        'cny': 'first',
        'mxn': 'first',
    })
    .sort_values('Date')
)

daily['Price_MA30'] = daily['Close_USD_60kg'].rolling(30, min_periods=1).mean()
daily['YearMonth'] = daily['Date'].dt.to_period('M').astype(str)

daily.head()

## 1. Long-Run Coffee Price Trend

In [ ]:
monthly_price = (
    daily.groupby('YearMonth', as_index=False)['Close_USD_60kg']
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_price['YearMonth'], monthly_price['Close_USD_60kg'], color='#4c956c', linewidth=2)
ax.set_title('Average Monthly Coffee Commodity Price')
ax.set_xlabel('Month')
ax.set_ylabel('USD per 60 kg')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

## 2. Recent Daily Price Trend With 30-Day Moving Average

In [ ]:
recent = daily.tail(365)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(recent['Date'], recent['Close_USD_60kg'], label='Daily close', alpha=0.55, color='#2f6690')
ax.plot(recent['Date'], recent['Price_MA30'], label='30-day moving average', linewidth=3, color='#d68c45')
ax.set_title('Recent Coffee Price Trend')
ax.set_xlabel('Date')
ax.set_ylabel('USD per 60 kg')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Correlation Heatmap

In [ ]:
corr_columns = [
    'Close_USD_60kg',
    'Temp_Max',
    'Temp_Min',
    'Humidity',
    'Solar_Radiation',
    'Precipitation_mm',
    'brl',
    'cny',
    'mxn',
]

corr = daily[corr_columns].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix: Coffee Price, Weather, and FX Variables')
plt.tight_layout()
plt.show()

## 4. Weather vs Coffee Price Scatterplots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_specs = [
    ('Temp_Min', 'Minimum Temperature vs Price', '#4c956c'),
    ('Temp_Max', 'Maximum Temperature vs Price', '#2f6690'),
    ('Humidity', 'Humidity vs Price', '#b56576'),
    ('Precipitation_mm', 'Precipitation vs Price', '#d68c45'),
]

for ax, (column, title, color) in zip(axes.flat, plot_specs):
    sns.regplot(data=daily, x=column, y='Close_USD_60kg', scatter_kws={'alpha': 0.25}, line_kws={'color': 'black'}, color=color, ax=ax)
    ax.set_title(title)
    ax.set_ylabel('USD per 60 kg')

plt.tight_layout()
plt.show()

## 5. Distribution of Weather Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
distribution_columns = [
    'Temp_Max',
    'Temp_Min',
    'Humidity',
    'Solar_Radiation',
    'Precipitation_mm',
    'Close_USD_60kg',
]

for ax, column in zip(axes.flat, distribution_columns):
    sns.histplot(df[column].dropna(), kde=True, ax=ax, color='#4c956c')
    ax.set_title(f'Distribution of {column}')

plt.tight_layout()
plt.show()

## 6. Location-Level Average Conditions

In [ ]:
location_summary = (
    df.groupby('Location', as_index=False)
    .agg({
        'Temp_Max': 'mean',
        'Humidity': 'mean',
        'Precipitation_mm': 'mean',
    })
)

top_wet = location_summary.nlargest(10, 'Precipitation_mm').sort_values('Precipitation_mm')

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_wet['Location'], top_wet['Precipitation_mm'], color='#2f6690')
ax.set_title('Top 10 Wettest Coffee-Growing Locations')
ax.set_xlabel('Average precipitation (mm)')
ax.set_ylabel('Location')
plt.tight_layout()
plt.show()

## 7. Spatial Snapshot of a Single Date

In [ ]:
snapshot_date = df['Date'].max()
snapshot = df[df['Date'] == snapshot_date].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=snapshot, x='LON', y='LAT', hue='Precipitation_mm', palette='YlGnBu', s=120, ax=axes[0])
axes[0].set_title(f'Precipitation by Location on {snapshot_date.date()}')

sns.scatterplot(data=snapshot, x='LON', y='LAT', hue='Temp_Max', palette='flare', s=120, ax=axes[1])
axes[1].set_title(f'Max Temperature by Location on {snapshot_date.date()}')

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

## 8. Optional: Compare Monthly Weather and Monthly Price

In [ ]:
monthly = (
    daily.assign(Month=daily['Date'].dt.to_period('M').astype(str))
    .groupby('Month', as_index=False)
    .agg({
        'Close_USD_60kg': 'mean',
        'Temp_Min': 'mean',
        'Precipitation_mm': 'mean',
    })
)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(monthly['Month'], monthly['Close_USD_60kg'], color='#2f6690', label='Coffee price')
ax1.set_ylabel('USD per 60 kg', color='#2f6690')
ax1.tick_params(axis='y', labelcolor='#2f6690')

ax2 = ax1.twinx()
ax2.plot(monthly['Month'], monthly['Precipitation_mm'], color='#d68c45', alpha=0.8, label='Precipitation')
ax2.set_ylabel('Average precipitation (mm)', color='#d68c45')
ax2.tick_params(axis='y', labelcolor='#d68c45')

ax1.set_title('Monthly Coffee Price vs Monthly Average Precipitation')
ax1.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()